In [2]:

%pip install -q google-genai pydantic pandas


Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import os

print("Current folder:", os.getcwd())
print("\nFiles available:")
print(os.listdir("."))

notes_df = pd.read_csv("followup_notes.csv")

print("\nDataset shape:", notes_df.shape)
print("\nColumns:", notes_df.columns.tolist())
display(notes_df.head(10))

Current folder: C:\Users\ARUMUGA PANDIAN AM\Downloads\CareLoop_Clinics_Capstone\Part_4_GenAI

Files available:
['.ipynb_checkpoints', '04_genai.ipynb', 'followup_notes.csv']

Dataset shape: (30, 3)

Columns: ['note_id', 'appointment_id', 'note_text']


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [2]:
import json
import time
import getpass
import pandas as pd

from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

print("Imports successful!")
print("Pandas version:", pd.__version__)

Imports successful!
Pandas version: 2.3.3


In [3]:
import getpass
from google import genai

API_KEY = getpass.getpass("Paste your Gemini API key here: ")

client = genai.Client(api_key=API_KEY)

print("Gemini API key loaded successfully!")

Paste your Gemini API key here:  ········


Gemini API key loaded successfully!


In [4]:
available_models = []

for model in client.models.list():
    if "flash" in model.name.lower() and "generateContent" in model.supported_actions:
        available_models.append(model.name)

print("Available Gemini Flash models:")
for model in available_models:
    print(model)

Available Gemini Flash models:
models/gemini-2.5-flash
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash


In [5]:
MODEL_NAME = "gemini-2.5-flash"

print("Selected model:", MODEL_NAME)

Selected model: gemini-2.5-flash


In [6]:
SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {"type": "STRING"},
        "confidence": {"type": "NUMBER"},
        "summary": {"type": "STRING"},
        "recommended_action": {"type": "STRING"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("JSON response schema created successfully!")

JSON response schema created successfully!


In [7]:
SYSTEM_TEXT = """
You are an expert healthcare operations analyst.

Your task is to analyse patient follow-up notes explaining
why a patient missed a clinic appointment.

Classify each note into exactly one of these root causes:

1. forgot_or_no_reminder
2. distance_or_transport
3. cost_or_fee
4. illness_or_emergency
5. work_or_schedule
6. other

Rules:
- Select only one root cause.
- Do not invent information.
- If the reason is unclear or does not fit the categories, use "other".
- Confidence must be a number between 0 and 1.
- Summary must contain no more than 25 words.
- Recommend one practical action for the clinic.
- Understand English and Hinglish notes.
- Return only the requested JSON structure.
"""

print("CRAFT system instructions created!")

CRAFT system instructions created!


In [8]:
def analyse_note(note_text):

    prompt = f"""
    Context: A clinic wants to understand why patients miss appointments.

    Role: You are a healthcare operations analyst.

    Action: Analyse this patient follow-up note:
    "{note_text}"

    Identify the most likely root cause and suggest a practical clinic action.

    Tone: Professional, concise and evidence-based.

    Format: Return JSON with exactly these fields:
    root_cause, confidence, summary, recommended_action.

    Use only the allowed root cause categories in the system instructions.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)

    return data


print("analyse_note() function created successfully!")

analyse_note() function created successfully!


In [9]:
test_note = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

result = analyse_note(test_note)

print("Original note:")
print(test_note)

print("\nGemini analysis:")
print(json.dumps(result, indent=4))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [10]:
MODEL_NAME = "gemini-3.8-flash"

print("Updated model:", MODEL_NAME)

Updated model: gemini-3.8-flash


In [11]:
test_note = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

result = analyse_note(test_note)

print("Original note:")
print(test_note)

print("\nGemini analysis:")
print(json.dumps(result, indent=4))

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Thinking level MINIMAL is not supported for this model. Please retry with other thinking level.', 'status': 'INVALID_ARGUMENT'}}

In [12]:
import getpass
from google import genai

API_KEY = getpass.getpass("Paste your Gemini API key here: ")

client = genai.Client(api_key=API_KEY)

print("Gemini API key loaded successfully!")

Paste your Gemini API key here:  ········


Gemini API key loaded successfully!


In [13]:
MODEL_NAME = "gemini-3.8-flash"

print("Updated model:", MODEL_NAME)

Updated model: gemini-3.8-flash


In [14]:
test_note = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

result = analyse_note(test_note)

print("Original note:")
print(test_note)

print("\nGemini analysis:")
print(json.dumps(result, indent=4))

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Thinking level MINIMAL is not supported for this model. Please retry with other thinking level.', 'status': 'INVALID_ARGUMENT'}}

In [15]:
def analyse_note(note_text):

    prompt = f"""
    Context: A clinic wants to understand why patients miss appointments.

    Role: You are a healthcare operations analyst.

    Action: Analyse this patient follow-up note:
    "{note_text}"

    Identify the most likely root cause and suggest a practical clinic action.

    Tone: Professional, concise and evidence-based.

    Format: Return JSON with exactly these fields:
    root_cause, confidence, summary, recommended_action.

    Use only the allowed root cause categories in the system instructions.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    data = json.loads(response.text)

    return data


print("Updated analyse_note() function successfully!")

Updated analyse_note() function successfully!


In [16]:
result = analyse_note(test_note)

print("Original note:")
print(test_note)

print("\nGemini analysis:")
print(json.dumps(result, indent=4))

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [17]:
MODEL_NAME = "gemini-3.5-flash"

print("Current model:", MODEL_NAME)

Current model: gemini-3.5-flash


In [18]:
result = analyse_note(test_note)

print("Original note:")
print(test_note)

print("\nGemini analysis:")
print(json.dumps(result, indent=4))

Original note:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Gemini analysis:
{
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.98,
    "summary": "Patient missed her appointment after not receiving a reminder message and forgetting the scheduled visit.",
    "recommended_action": "Verify patient contact details and audit automated reminder delivery logs to ensure notifications are sending."
}


In [19]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

print("Pydantic validation model created!")

Pydantic validation model created!


In [20]:
validated_result = FollowupInsight(**result)

if validated_result.root_cause not in ALLOWED_CAUSES:
    print("Invalid root cause")
else:
    print("N001 validation successful!")
    print(validated_result.model_dump())

N001 validation successful!
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed her appointment after not receiving a reminder message and forgetting the scheduled visit.', 'recommended_action': 'Verify patient contact details and audit automated reminder delivery logs to ensure notifications are sending.'}


In [21]:
def analyse_note_with_raw(note_text):

    prompt = f"""
    Context: A clinic wants to understand why patients miss appointments.

    Role: You are a healthcare operations analyst.

    Action: Analyse this patient follow-up note:
    "{note_text}"

    Identify the most likely root cause and suggest a practical clinic action.

    Tone: Professional, concise and evidence-based.

    Format: Return JSON with exactly these fields:
    root_cause, confidence, summary, recommended_action.

    Use only the allowed root cause categories in the system instructions.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw_text = response.text
    data = json.loads(raw_text)

    return data, raw_text


print("Raw-response analysis function created!")

Raw-response analysis function created!


In [22]:
results = []
api_call_count = 0

for _, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"\nProcessing {note_id}...")

    # Handle blank notes without calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":

        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        }

        results.append(record)
        print("Skipped: blank note")
        continue

    try:

        # Reuse the successful N001 result
        if note_id == "N001":
            data = result
            raw_text = json.dumps(result, ensure_ascii=False)

        else:
            if api_call_count > 0:
                time.sleep(13)

            data, raw_text = analyse_note_with_raw(note_text)
            api_call_count += 1

        # Preserve any fields returned by Gemini
        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_text
        }

        try:
            validated = FollowupInsight(**data)

            if validated.root_cause in ALLOWED_CAUSES:
                record["validation_status"] = "valid"

        except (ValidationError, ValueError):
            record["validation_status"] = "invalid"

        results.append(record)

        print("Root cause:", record["root_cause"])
        print("Validation:", record["validation_status"])

    except Exception as e:

        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "",
            "confidence": "",
            "summary": "",
            "recommended_action": "",
            "validation_status": "api_error",
            "raw_response": f"API_ERROR: {str(e)}"
        }

        results.append(record)
        print("API error:", str(e))


# Create and save the output
notes_analysed_df = pd.DataFrame(results)

output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

notes_analysed_df = notes_analysed_df[output_columns]

notes_analysed_df.to_csv("notes_analysed.csv", index=False)

print("\n========== TASK 4.2 SUMMARY ==========")
print("Total notes processed:", len(notes_analysed_df))
print("New Gemini API calls:", api_call_count)
print("\nValidation status:")
print(notes_analysed_df["validation_status"].value_counts())
print("\nSaved: notes_analysed.csv")

display(notes_analysed_df)


Processing N001...
Root cause: forgot_or_no_reminder
Validation: valid

Processing N002...
Root cause: distance_or_transport
Validation: valid

Processing N003...
Root cause: cost_or_fee
Validation: valid

Processing N004...
Skipped: blank note

Processing N005...
Root cause: illness_or_emergency
Validation: valid

Processing N006...
Root cause: work_or_schedule
Validation: valid

Processing N007...
Root cause: forgot_or_no_reminder
Validation: valid

Processing N008...
Root cause: distance_or_transport
Validation: valid

Processing N009...
Root cause: other
Validation: valid

Processing N010...
Root cause: work_or_schedule
Validation: valid

========== TASK 4.2 SUMMARY ==========
Total notes processed: 10
New Gemini API calls: 8

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Saved: notes_analysed.csv


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,Patient missed her appointment after not recei...,Verify patient contact details and audit autom...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi..."
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they li...,Offer telehealth options for routine check-ups...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,0.98,Patient skipped the appointment due to high co...,Provide information on discounted follow-up pa...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.98,..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.95,Patient was unable to attend due to high fever...,Offer a telehealth consultation or facilitate ...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgen...,Offer flexible evening or weekend appointment ...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment after not recei...,Audit and optimize the automated SMS reminder ...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
7,N008,A0222,distance_or_transport,0.98,Patient missed the appointment due to a city b...,Offer telehealth consultation options or assis...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,other,0.90,Patient could not be reached after three attem...,Attempt contact via alternative channels like ...,valid,"{""root_cause"":""other"",""confidence"":0.9,""summar..."
9,N010,A0596,work_or_schedule,0.95,Patient missed appointment due to a factory sh...,Offer Saturday or evening appointment slots to...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."


In [23]:
bad_replies = [
    {
        "root_cause": "forgot_or_no_reminder",
        "confidence": 1.7,
        "summary": "Patient forgot the appointment.",
        "recommended_action": "Send reminders."
    },
    {
        "root_cause": "other",
        "confidence": 0.8,
        "summary": "The reason is unclear."
        # recommended_action is intentionally missing
    }
]

for i, bad_reply in enumerate(bad_replies, start=1):

    print(f"\nTesting bad reply {i}:")

    try:
        FollowupInsight(**bad_reply)
        print("Unexpected: validation passed")

    except ValidationError as e:
        print("Validation correctly caught the error!")
        print(e)


Testing bad reply 1:
Validation correctly caught the error!
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Testing bad reply 2:
Validation correctly caught the error!
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...The reason is unclear.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [24]:
def analyse_with_prompt(note_text, prompt_text):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text.format(note_text=note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw_text = response.text
    data = json.loads(raw_text)

    return data, raw_text


# Get the original N007 and N009 notes
test_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
].set_index("note_id")

# Simple V1 prompt
V1_PROMPT = "Classify this note: {note_text}"

# Detailed V2 CRAFT prompt
V2_PROMPT = """
Context: A clinic wants to understand why patients miss appointments.

Role: You are a healthcare operations analyst.

Action: Analyse this patient follow-up note:
"{note_text}"

Identify the most likely root cause and suggest a practical clinic action.

Tone: Professional, concise and evidence-based.

Format: Return JSON with exactly these fields:
root_cause, confidence, summary, recommended_action.

Use only the allowed root cause categories in the system instructions.
"""

comparison_results = []

for note_id in ["N007", "N009"]:

    note_text = test_notes.loc[note_id, "note_text"]

    print(f"\nTesting {note_id} with V1 simple prompt...")

    v1_data, v1_raw = analyse_with_prompt(
        note_text, V1_PROMPT
    )

    time.sleep(13)

    # Reuse the original CRAFT response from Task 4.2
    existing_v2 = notes_analysed_df[
        notes_analysed_df["note_id"] == note_id
    ].iloc[0]

    v2_data = {
        "root_cause": existing_v2["root_cause"],
        "confidence": existing_v2["confidence"],
        "summary": existing_v2["summary"],
        "recommended_action": existing_v2["recommended_action"]
    }

    for version, data in [
        ("V1 Simple", v1_data),
        ("V2 CRAFT", v2_data)
    ]:

        try:
            validated = FollowupInsight(**data)

            status = (
                "valid"
                if validated.root_cause in ALLOWED_CAUSES
                else "invalid_category"
            )

        except ValidationError:
            status = "invalid"

        comparison_results.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "validation_status": status
        })

    print("V1:", v1_data["root_cause"])
    print("V2:", v2_data["root_cause"])


comparison_df = pd.DataFrame(comparison_results)

print("\nPROMPT COMPARISON")
display(comparison_df)

print("New API calls in this comparison: 2")


Testing N007 with V1 simple prompt...
V1: forgot_or_no_reminder
V2: forgot_or_no_reminder

Testing N009 with V1 simple prompt...
V1: other
V2: other

PROMPT COMPARISON


,note_id,prompt_version,root_cause,confidence,validation_status
0,N007,V1 Simple,forgot_or_no_reminder,0.98,valid
1,N007,V2 CRAFT,forgot_or_no_reminder,0.95,valid
2,N009,V1 Simple,other,0.90,valid
3,N009,V2 CRAFT,other,0.90,valid


New API calls in this comparison: 2


In [25]:

import pandas as pd
from pathlib import Path

# Load the two datasets
notes_df = pd.read_csv("notes_analysed.csv")

appointments_df = pd.read_csv(
    "../Part_1_Data_Foundations/appointments_clean.csv"
)

# Merge using appointment_id
merged_df = notes_df.merge(
    appointments_df,
    on="appointment_id",
    how="left"
)

# Display important columns
print("MERGED DATASET")
print("Shape:", merged_df.shape)
print("\nSelected columns:")
display(
    merged_df[
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "confidence",
            "reminder_sent",
            "no_show"
        ]
    ]
)

# Check merge quality
print("\nMERGE VALIDATION")
print("Total notes:", len(notes_df))
print("Merged rows:", len(merged_df))
print("Missing appointment matches:", merged_df["reminder_sent"].isna().sum())

# Cross-tabulation: root cause vs reminder status
print("\nROOT CAUSE vs REMINDER STATUS")
root_cause_reminder = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    margins=True
)

display(root_cause_reminder)

# Important acceptance checks
print("\nACCEPTANCE CHECKS")
print("Exactly 10 merged rows:", len(merged_df) == 10)

n001_n007 = merged_df[
    merged_df["note_id"].isin(["N001", "N007"])
]

print(
    "N001 and N007 both have reminder_sent = No:",
    len(n001_n007) == 2
    and n001_n007["reminder_sent"].astype(str).str.strip().str.lower().eq("no").all()
)

no_reminder_count = (
    merged_df["reminder_sent"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("no")
    .sum()
)

print("Notes with reminder_sent = No:", no_reminder_count)
print("Expected count is 5:", no_reminder_count == 5)


MERGED DATASET
Shape: (10, 23)

Selected columns:


,note_id,appointment_id,root_cause,confidence,reminder_sent,no_show
0,N001,A0872,forgot_or_no_reminder,0.98,No,1
1,N002,A0897,distance_or_transport,0.95,No,1
2,N003,A0526,cost_or_fee,0.98,Yes,1
3,N004,A0926,other,0.00,No,1
4,N005,A0534,illness_or_emergency,0.95,Yes,1
5,N006,A0507,work_or_schedule,0.95,Yes,1
6,N007,A0333,forgot_or_no_reminder,0.95,No,1
7,N008,A0222,distance_or_transport,0.98,Yes,1
8,N009,A0977,other,0.90,Yes,1
9,N010,A0596,work_or_schedule,0.95,No,1



MERGE VALIDATION
Total notes: 10
Merged rows: 10
Missing appointment matches: 0

ROOT CAUSE vs REMINDER STATUS


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10



ACCEPTANCE CHECKS
Exactly 10 merged rows: True
N001 and N007 both have reminder_sent = No: True
Notes with reminder_sent = No: 5
Expected count is 5: True


In [26]:

from pathlib import Path

readme_content = r"""
# CareLoop Clinics — No-Show Guard

**GitHub Repository:** [Add your public GitHub repository URL here]

## Project Overview

CareLoop Clinics — No-Show Guard is a data analytics and machine learning project
that studies patient appointment no-shows across eight clinics. It combines data
cleaning, SQL analytics, predictive modelling and Generative AI analysis of
patient follow-up notes to suggest practical ways to reduce missed appointments.

## Key Findings

1. **Part 1 — Data Foundations:** Cleaned 990 appointments from an initial
   dataset of 1,020 records. The observed no-show rate was 22.02%.

2. **Part 2 — SQL Analytics:** Clinic C08 had the highest no-show rate at
   33.33%. Appointments booked 31–45 days ahead had a 60.47% no-show rate
   (based on 43 appointments).

3. **Part 3 — Machine Learning:** The tuned Random Forest achieved a test
   ROC-AUC of 0.7075 and a test F1 score of 0.4643. Its best cross-validation
   F1 score was 0.5047.

4. **Part 4 — Generative AI:** Gemini analysed 9 nonblank follow-up notes
   successfully, while 1 blank note was skipped. Both notes classified as
   forgot_or_no_reminder belonged to appointments where no reminder was sent.

## Business Recommendation

CareLoop Clinics should introduce a targeted appointment-retention programme.
The overall no-show rate is 22.02% across 990 cleaned appointments, while clinic
C08 has a particularly high rate of 33.33%, making it a priority for operational
review. Long booking lead times also deserve attention: the 31–45-day group had
a 60.47% no-show rate across 43 appointments. The tuned Random Forest provides
a useful risk-ranking starting point, with a test ROC-AUC of 0.7075, although
its test F1 score of 0.4643 shows that predictions still need improvement.
In the 10-note GenAI illustration, both patients classified as
forgot_or_no_reminder had received no reminder, and 5 of the 10 appointments
had no reminder recorded. Clinics should therefore audit reminder delivery,
confirm patient contact details, prioritise follow-up for higher-risk
appointments, and investigate transport, cost, illness and scheduling barriers.
These actions should be tested through a monitored pilot before wider rollout.

## Selected Machine Learning Model

**Selected model:** Tuned Random Forest Classifier.

The tuned Random Forest was selected because it achieved the best
cross-validation F1 score of 0.5047 among the evaluated models after
hyperparameter tuning. On the held-out test set, it achieved:

- Accuracy: 0.6970
- Precision: 0.3824
- Recall: 0.5909
- F1 score: 0.4643
- ROC-AUC: 0.7075

The classification threshold was 0.50, the default probability threshold.
The model is intended as a decision-support tool, not as a definitive
prediction of an individual patient's behaviour.

## Limitations

1. **Limited GenAI sample:** Only 10 follow-up notes were analysed. This is
   an illustration and cannot establish the true distribution of reasons
   for missed appointments across all clinics.

2. **Patient overlap:** The random train/test split may place appointments
   belonging to the same patient in both sets. This can make evaluation
   results more optimistic than performance on entirely unseen patients.

3. **Random rather than temporal split:** The model was evaluated using a
   random split rather than training on earlier dates and testing on later
   dates. Real-world future performance may therefore differ.

## Technology Stack

- Python
- Pandas and NumPy
- Matplotlib and Seaborn
- Scikit-learn
- MySQL
- Google Gemini API
- Pydantic
- Jupyter Notebook

## Project Structure

- `Part_1_Data_Foundations/` — ingestion, cleaning and weather enrichment
- `Part_2_SQL_Analytics/` — SQL database and analytical queries
- `Part_3_Machine_Learning/` — model training, evaluation and saved outputs
- `Part_4_GenAI/` — Gemini note analysis, validation and recommendations

## Responsible Use

The model and GenAI outputs should support clinic staff rather than replace
human judgement. Patient follow-up recommendations should be reviewed by
authorised staff, and API keys or other secrets must never be committed
to the repository.
"""

readme_path = Path("README.md")
readme_path.write_text(readme_content.strip() + "\n", encoding="utf-8")

print("README.md created successfully!")
print("Saved at:", readme_path.resolve())
print("File size:", readme_path.stat().st_size, "bytes")


README.md created successfully!
Saved at: C:\Users\ARUMUGA PANDIAN AM\Downloads\CareLoop_Clinics_Capstone\Part_4_GenAI\README.md
File size: 4280 bytes


In [27]:

from pathlib import Path
import re

# Project root is one folder above Part_4_GenAI
project_root = Path("..").resolve()

print("PROJECT ROOT:", project_root)

print("\n1. PROJECT FILES")
for path in sorted(project_root.rglob("*")):
    if any(part in {".ipynb_checkpoints", "__pycache__", ".git"} for part in path.parts):
        continue
    if path.is_file():
        print(path.relative_to(project_root))

# Check text-based files and notebooks for common Google API key patterns
print("\n2. API KEY SECURITY CHECK")

key_pattern = re.compile(r"AIza[0-9A-Za-z_-]{30,}")
text_extensions = {
    ".ipynb", ".py", ".sql", ".md", ".txt",
    ".json", ".yaml", ".yml", ".env", ".ini"
}

exposed_key_files = []
env_files = []

for path in project_root.rglob("*"):
    if not path.is_file():
        continue

    if any(part in {".ipynb_checkpoints", "__pycache__", ".git"} for part in path.parts):
        continue

    if path.name == ".env" or path.suffix.lower() == ".env":
        env_files.append(str(path.relative_to(project_root)))

    if path.suffix.lower() in text_extensions or path.name == ".env":
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
            if key_pattern.search(content):
                exposed_key_files.append(str(path.relative_to(project_root)))
        except Exception:
            pass

if exposed_key_files:
    print("WARNING: Possible Google API key found in:")
    for name in exposed_key_files:
        print("-", name)
else:
    print("No Google API key pattern detected in scanned files.")

print("\n.env files found:", env_files if env_files else "None")

# Check essential deliverables
print("\n3. IMPORTANT FILE CHECKS")

required_files = [
    "Part_1_Data_Foundations/appointments_clean.csv",
    "Part_1_Data_Foundations/clinics_enriched.csv",
    "Part_1_Data_Foundations/clinic_daily_weather.csv",
    "Part_2_SQL_Analytics",
    "Part_3_Machine_Learning/outputs/best_no_show_model.pkl",
    "Part_3_Machine_Learning/outputs/model_comparison.csv",
    "Part_3_Machine_Learning/outputs/test_risk_scores.csv",
    "Part_4_GenAI/notes_analysed.csv",
    "Part_4_GenAI/README.md",
    "Part_4_GenAI/04_genai.ipynb"
]

for item in required_files:
    path = project_root / item
    print(("FOUND  " if path.exists() else "MISSING"), item)

print("\nAudit completed. Review any warnings before publishing.")


PROJECT ROOT: C:\Users\ARUMUGA PANDIAN AM\Downloads\CareLoop_Clinics_Capstone

1. PROJECT FILES
03_model.ipynb.ipynb
Part_1_Data_Foundations\appointments_clean.csv
Part_1_Data_Foundations\clinic_daily_weather.csv
Part_1_Data_Foundations\clinics_enriched.csv
Part_1_Data_Foundations\Part_1_Data_Foundations.sql
Part_1_Data_Foundations\rejected_rows.json
Part_1_Data_Foundations\unknown_outcomes.csv
Part_2_SQL_Analytics\Part_2_SQL_Analytics.sql
Part_3_Machine_Learning\outputs\best_no_show_model.pkl
Part_3_Machine_Learning\outputs\model_comparison.csv
Part_3_Machine_Learning\outputs\test_risk_scores.csv
Part_4_GenAI\04_genai.ipynb
Part_4_GenAI\followup_notes.csv
Part_4_GenAI\notes_analysed.csv
Part_4_GenAI\README.md
Untitled.ipynb

2. API KEY SECURITY CHECK
No Google API key pattern detected in scanned files.

.env files found: None

3. IMPORTANT FILE CHECKS
FOUND   Part_1_Data_Foundations/appointments_clean.csv
FOUND   Part_1_Data_Foundations/clinics_enriched.csv
FOUND   Part_1_Data_Foundat

In [28]:

from pathlib import Path
import json

root = Path("..").resolve()

notebooks = [
    root / "03_model.ipynb.ipynb",
    root / "Untitled.ipynb",
    root / "Part_4_GenAI" / "04_genai.ipynb"
]

print("NOTEBOOK INSPECTION\n")

for path in notebooks:
    print("=" * 60)
    print("File:", path.name)
    print("Exists:", path.exists())

    if path.exists():
        print("Size:", round(path.stat().st_size / 1024, 2), "KB")

        try:
            with open(path, "r", encoding="utf-8") as f:
                notebook = json.load(f)

            cells = notebook.get("cells", [])
            code_cells = sum(c.get("cell_type") == "code" for c in cells)
            markdown_cells = sum(c.get("cell_type") == "markdown" for c in cells)

            print("Total cells:", len(cells))
            print("Code cells:", code_cells)
            print("Markdown cells:", markdown_cells)

            for cell in cells[:3]:
                source = "".join(cell.get("source", []))
                print("Preview:", source[:180].replace("\n", " "))

        except Exception as e:
            print("Could not inspect:", e)

print("\nINSPECTION COMPLETE")


NOTEBOOK INSPECTION

File: 03_model.ipynb.ipynb
Exists: True
Size: 39.28 KB
Total cells: 16
Code cells: 16
Markdown cells: 0
Preview:  import pandas as pd import numpy as np import os  # Load dataset from Part 1 folder file_path = os.path.join(     "..",     "Part_1_Data_Foundations",     "appointments_clean.csv"
Preview:  from sklearn.model_selection import train_test_split  # Select input features feature_columns = [     "clinic_id",     "age",     "gender",     "distance_km",     "fee_inr",     "
Preview:  from sklearn.model_selection import train_test_split  # Select input features feature_columns = [     "clinic_id",     "age",     "gender",     "distance_km",     "fee_inr",     "
File: Untitled.ipynb
Exists: True
Size: 0.33 KB
Total cells: 1
Code cells: 1
Markdown cells: 0
Preview: 
File: 04_genai.ipynb
Exists: True
Size: 148.25 KB
Total cells: 28
Code cells: 28
Markdown cells: 0
Preview:  %pip install -q google-genai pydantic pandas 
Preview: import pandas as pd import os  print

In [29]:

from pathlib import Path
import shutil

root = Path("..").resolve()

source = root / "03_model.ipynb.ipynb"
destination = root / "Part_3_Machine_Learning" / "03_model.ipynb"

if source.exists() and not destination.exists():
    shutil.move(str(source), str(destination))
    print("ML notebook moved and renamed successfully!")
elif destination.exists():
    print("The correctly named ML notebook already exists.")
else:
    print("ERROR: Source notebook not found.")

print("\nDestination:", destination)
print("File exists:", destination.exists())


ML notebook moved and renamed successfully!

Destination: C:\Users\ARUMUGA PANDIAN AM\Downloads\CareLoop_Clinics_Capstone\Part_3_Machine_Learning\03_model.ipynb
File exists: True


In [30]:

from pathlib import Path

root = Path("..").resolve()

master_readme = """# CareLoop Clinics — No-Show Guard

**A Data Science, Machine Learning and Generative AI Capstone Project**

## Overview

CareLoop Clinics — No-Show Guard analyses patient appointment no-shows across eight clinics. The project combines Python data engineering, SQL analytics, machine learning prediction and Gemini-powered analysis of patient follow-up notes.

## Project Results

| Part | Work completed | Key result |
|---|---|---|
| Part 1 — Data Foundations | Data cleaning and weather enrichment | 990 cleaned appointments; 22.02% no-show rate |
| Part 2 — SQL Analytics | Clinic and appointment analysis | C08 highest no-show rate: 33.33% |
| Part 3 — Machine Learning | Three models, cross-validation and tuning | Tuned Random Forest test ROC-AUC: 0.7075 |
| Part 4 — Generative AI | Gemini note classification and Pydantic validation | 9 nonblank notes analysed; 1 blank note skipped |

## Business Objective

Identify appointment no-show patterns, predict patients at risk of missing appointments, understand reasons from follow-up notes, and recommend practical clinic interventions.

## Repository Structure

```text
CareLoop_Clinics_Capstone/
│
├── Part_1_Data_Foundations/
│   ├── appointments_clean.csv
│   ├── clinics_enriched.csv
│   ├── clinic_daily_weather.csv
│   ├── rejected_rows.json
│   ├── unknown_outcomes.csv
│   └── Part_1_Data_Foundations.sql
│
├── Part_2_SQL_Analytics/
│   └── Part_2_SQL_Analytics.sql
│
├── Part_3_Machine_Learning/
│   ├── 03_model.ipynb
│   └── outputs/
│       ├── best_no_show_model.pkl
│       ├── model_comparison.csv
│       └── test_risk_scores.csv
│
└── Part_4_GenAI/
    ├── 04_genai.ipynb
    ├── followup_notes.csv
    ├── notes_analysed.csv
    └── README.md
```

## Selected Model

The tuned Random Forest was selected based on the best cross-validation F1 score of 0.5047. Its held-out test results were:

- Accuracy: 0.6970
- Precision: 0.3824
- Recall: 0.5909
- F1 score: 0.4643
- ROC-AUC: 0.7075
- Classification threshold: 0.50

## Technology Stack

Python, Pandas, NumPy, Scikit-learn, MySQL, Google Gemini API, Pydantic and Jupyter Notebook.

## Limitations

- Only 10 follow-up notes were analysed by GenAI; this is not representative of all clinic no-show reasons.
- The random train/test split may place appointments from the same patient in both sets.
- A random split does not fully simulate predicting future appointments from past data.

## Responsible Use

Predictions are intended to support clinic staff, not replace human judgement. GenAI classifications should be reviewed before operational decisions. API keys and credentials must never be committed to this repository.

## Detailed GenAI Report

See [Part 4 README](Part_4_GenAI/README.md) for the detailed business recommendation and GenAI results.
"""

readme_path = root / "README.md"
readme_path.write_text(master_readme, encoding="utf-8")

print("Master README created!")
print("Location:", readme_path)
print("Size:", readme_path.stat().st_size, "bytes")


Master README created!
Location: C:\Users\ARUMUGA PANDIAN AM\Downloads\CareLoop_Clinics_Capstone\README.md
Size: 3026 bytes


In [31]:

from pathlib import Path

root = Path("..").resolve()

expected_files = [
    "README.md",
    "Part_1_Data_Foundations/appointments_clean.csv",
    "Part_1_Data_Foundations/clinics_enriched.csv",
    "Part_1_Data_Foundations/clinic_daily_weather.csv",
    "Part_1_Data_Foundations/rejected_rows.json",
    "Part_1_Data_Foundations/unknown_outcomes.csv",
    "Part_1_Data_Foundations/Part_1_Data_Foundations.sql",
    "Part_2_SQL_Analytics/Part_2_SQL_Analytics.sql",
    "Part_3_Machine_Learning/03_model.ipynb",
    "Part_3_Machine_Learning/outputs/best_no_show_model.pkl",
    "Part_3_Machine_Learning/outputs/model_comparison.csv",
    "Part_3_Machine_Learning/outputs/test_risk_scores.csv",
    "Part_4_GenAI/04_genai.ipynb",
    "Part_4_GenAI/followup_notes.csv",
    "Part_4_GenAI/notes_analysed.csv",
    "Part_4_GenAI/README.md"
]

print("FINAL SUBMISSION FILE CHECK")
print("=" * 55)

missing = []

for item in expected_files:
    path = root / item
    if path.exists():
        print("FOUND  ", item)
    else:
        print("MISSING", item)
        missing.append(item)

print("\nTotal expected files:", len(expected_files))
print("Files found:", len(expected_files) - len(missing))
print("Files missing:", len(missing))

if not missing:
    print("\nALL EXPECTED FILES ARE PRESENT!")
else:
    print("\nPlease review the missing files.")


FINAL SUBMISSION FILE CHECK
FOUND   README.md
FOUND   Part_1_Data_Foundations/appointments_clean.csv
FOUND   Part_1_Data_Foundations/clinics_enriched.csv
FOUND   Part_1_Data_Foundations/clinic_daily_weather.csv
FOUND   Part_1_Data_Foundations/rejected_rows.json
FOUND   Part_1_Data_Foundations/unknown_outcomes.csv
FOUND   Part_1_Data_Foundations/Part_1_Data_Foundations.sql
FOUND   Part_2_SQL_Analytics/Part_2_SQL_Analytics.sql
FOUND   Part_3_Machine_Learning/03_model.ipynb
FOUND   Part_3_Machine_Learning/outputs/best_no_show_model.pkl
FOUND   Part_3_Machine_Learning/outputs/model_comparison.csv
FOUND   Part_3_Machine_Learning/outputs/test_risk_scores.csv
FOUND   Part_4_GenAI/04_genai.ipynb
FOUND   Part_4_GenAI/followup_notes.csv
FOUND   Part_4_GenAI/notes_analysed.csv
FOUND   Part_4_GenAI/README.md

Total expected files: 16
Files found: 16
Files missing: 0

ALL EXPECTED FILES ARE PRESENT!
